# Strava Leistungsanalyse aus FIT-Dateien

Dieses Notebook ist für **Visual Studio Code + Jupyter** ausgelegt und analysiert eine oder mehrere `.fit`-Dateien aus `data/raw/`.

Enthalten sind jetzt zusätzlich:

- Herzfrequenzzonen
- Leistungszonen / FTP-Schätzung
- Vergleich mehrerer FIT-Dateien


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / 'src'))

from fit_loader import find_fit_files, find_latest_fit_file, load_fit_records, load_fit_session_summary, load_multiple_fit_activities
from analysis import (
    add_time_bins,
    compare_activities,
    correlation_matrix,
    estimate_ftp,
    heart_rate_zones,
    power_zones,
    rolling_best_efforts,
    summarize_activity,
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
pd.set_option('display.max_columns', 100)


## Einstellungen
Passe diese Werte bei Bedarf an. Wenn `FTP = None` ist, wird eine FTP-Schätzung aus der besten 20-Minuten-Leistung berechnet.


In [ ]:
HR_MAX = 190
FTP = None  # z. B. 260


In [ ]:
fit_files = find_fit_files(PROJECT_ROOT / 'data' / 'raw')
print('Gefundene FIT-Dateien:')
for file in fit_files:
    print('-', file.name)

latest_fit_file = find_latest_fit_file(PROJECT_ROOT / 'data' / 'raw')
print(f'
Neueste Datei: {latest_fit_file.name}')


## Einzelanalyse der neuesten FIT-Datei


In [ ]:
fit_file = latest_fit_file
df = load_fit_records(fit_file)
session_summary = load_fit_session_summary(fit_file)
df = add_time_bins(df)

print(f'Records geladen: {len(df):,}')
df.head()


In [ ]:
summary_df = summarize_activity(df, session_summary)
summary_df


## Zeitverläufe


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)

if 'speed_kmh' in df.columns:
    axes[0].plot(df['elapsed_minutes'], df['speed_kmh'], color='tab:blue')
    axes[0].set_ylabel('km/h')
    axes[0].set_title('Geschwindigkeit')
else:
    axes[0].set_visible(False)

if 'heart_rate' in df.columns:
    axes[1].plot(df['elapsed_minutes'], df['heart_rate'], color='tab:red')
    axes[1].set_ylabel('bpm')
    axes[1].set_title('Herzfrequenz')
else:
    axes[1].set_visible(False)

if 'power' in df.columns:
    axes[2].plot(df['elapsed_minutes'], df['power'], color='tab:green')
    axes[2].set_ylabel('Watt')
    axes[2].set_title('Leistung')
else:
    axes[2].set_visible(False)

axes[2].set_xlabel('Zeit (min)')
plt.tight_layout()
plt.show()


In [ ]:
if 'altitude_m' in df.columns:
    plt.figure(figsize=(15, 4))
    plt.plot(df['elapsed_minutes'], df['altitude_m'], color='saddlebrown')
    plt.title('Höhenprofil')
    plt.xlabel('Zeit (min)')
    plt.ylabel('Höhe (m)')
    plt.tight_layout()
    plt.show()
else:
    print('Keine Höhendaten vorhanden.')


## Herzfrequenzzonen


In [ ]:
hr_zones_df = heart_rate_zones(df, hr_max=HR_MAX)
hr_zones_df


In [ ]:
if not hr_zones_df.empty:
    plt.figure(figsize=(10, 4))
    sns.barplot(data=hr_zones_df, x='Zone', y='Anteil (%)', palette='Reds')
    plt.title('Zeitanteil je Herzfrequenzzone')
    plt.tight_layout()
    plt.show()
else:
    print('Keine Herzfrequenzdaten für Zonenanalyse vorhanden.')


## Leistungszonen und FTP-Auswertung


In [ ]:
ftp_info = estimate_ftp(df)
ftp_info


In [ ]:
ftp_to_use = FTP if FTP is not None else ftp_info.get('ftp_estimated')
print(f'Verwendete FTP: {ftp_to_use}')
power_zones_df = power_zones(df, ftp=ftp_to_use)
power_zones_df


In [ ]:
if not power_zones_df.empty:
    plt.figure(figsize=(10, 4))
    sns.barplot(data=power_zones_df, x='Zone', y='Anteil (%)', palette='Greens')
    plt.title('Zeitanteil je Leistungszone')
    plt.tight_layout()
    plt.show()
else:
    print('Keine Leistungsdaten oder keine ausreichende FTP-Basis vorhanden.')


## Verteilungen und Zusammenhänge


In [ ]:
if 'heart_rate' in df.columns:
    plt.figure(figsize=(10, 4))
    sns.histplot(df['heart_rate'].dropna(), bins=25, kde=True, color='tab:red')
    plt.title('Verteilung Herzfrequenz')
    plt.xlabel('bpm')
    plt.tight_layout()
    plt.show()
else:
    print('Keine Herzfrequenzdaten vorhanden.')


In [ ]:
corr = correlation_matrix(df, ['speed_kmh', 'heart_rate', 'power', 'cadence', 'altitude_m'])
if not corr.empty:
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
    plt.title('Korrelation ausgewählter Kennzahlen')
    plt.tight_layout()
    plt.show()
else:
    print('Nicht genügend Daten für eine Korrelationsmatrix vorhanden.')


## Bestleistungen


In [ ]:
best_power = rolling_best_efforts(df, column='power', windows=(5, 30, 60, 300, 1200))
if not best_power.empty:
    display(best_power)

    plt.figure(figsize=(10, 4))
    plt.plot(best_power['Fenster (s)'], best_power['Bestwert'], marker='o')
    plt.xscale('log')
    plt.title('Bestleistungen Leistung')
    plt.xlabel('Fenster (Sekunden, logarithmisch)')
    plt.ylabel('Watt')
    plt.tight_layout()
    plt.show()
else:
    print('Keine Leistungsdaten für Bestleistungen vorhanden.')


## Vergleich mehrerer FIT-Dateien


In [ ]:
activity_map = {name: add_time_bins(frame) for name, frame in load_multiple_fit_activities(PROJECT_ROOT / 'data' / 'raw').items()}
comparison_df = compare_activities(activity_map)
comparison_df


In [ ]:
if len(activity_map) > 1:
    plt.figure(figsize=(14, 5))
    for name, activity_df in activity_map.items():
        if 'elapsed_minutes' in activity_df.columns and 'speed_kmh' in activity_df.columns:
            plt.plot(activity_df['elapsed_minutes'], activity_df['speed_kmh'], label=name, alpha=0.8)
    plt.title('Vergleich Geschwindigkeitsverlauf mehrerer FIT-Dateien')
    plt.xlabel('Zeit (min)')
    plt.ylabel('Geschwindigkeit (km/h)')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Für den Vergleich werden mindestens zwei FIT-Dateien benötigt.')


In [ ]:
if len(activity_map) > 1 and 'Distanz (km)' in comparison_df.columns and 'Ø Leistung (W)' in comparison_df.columns:
    plot_df = comparison_df.dropna(subset=['Distanz (km)', 'Ø Leistung (W)'])
    if not plot_df.empty:
        plt.figure(figsize=(10, 5))
        sns.scatterplot(data=plot_df, x='Distanz (km)', y='Ø Leistung (W)', hue='Datei', s=120)
        plt.title('Vergleich Distanz vs. Durchschnittsleistung')
        plt.tight_layout()
        plt.show()
    else:
        print('Nicht genügend Leistungsdaten für den Vergleich vorhanden.')
else:
    print('Kein Leistungs-Vergleich möglich.')


## Nächste Schritte

Mögliche weitere Erweiterungen:

- individuelle Herzfrequenz- und Leistungszonen
- automatische Berichte als HTML/PDF
- Kartenansicht mit GPS-Daten
- Filter nach Monat, Strecke oder Aktivitätstyp
